# CIFAR-10 CFG strength sweep

Load a class-conditional CIFAR-10 flow-matching UNet from MLflow, then compare samples across classifier-free guidance strengths. Set RUN_ID if you want a specific run; otherwise the notebook picks the lowest best_mse run in the experiment, falling back to the newest run.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import torch
from IPython.display import display
from mlflow.tracking import MlflowClient

from models.ode_solvers import get_ode_solver_from_name, sample_conditional

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# MLflow server / run selection.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "Flow Matching CIFAR10 Conditional"
RUN_ID = None  # Set to a run id string to bypass auto-selection.
MODEL_ARTIFACT = "ClassCondUNet"

# Sampling controls.
CFG_SCALES = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
N_CLASSES = 10
IMAGE_SHAPE = (3, 32, 32)
ODE_SOLVER_NAME = "euler_solver"
ODE_STEPS = 50
SEED = 0
CLAMP_MODE = "clamp"

CIFAR10_LABELS = [
    "airplane",
    "auto",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()


def select_run_id(experiment_name: str, run_id: str | None = None):
    if run_id is not None:
        run = client.get_run(run_id)
        return run.info.run_id, run

    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"MLflow experiment not found: {experiment_name!r}")

    runs = client.search_runs(
        [experiment.experiment_id],
        order_by=["attributes.start_time DESC"],
        max_results=100,
    )
    runs = [run for run in runs if run.info.status == "FINISHED"]
    if not runs:
        raise ValueError(f"No finished runs found in experiment: {experiment_name!r}")

    def sort_key(run):
        best_mse = run.data.metrics.get("best_mse", float("inf"))
        return (float(best_mse), -(run.info.start_time or 0))

    run = sorted(runs, key=sort_key)[0]
    return run.info.run_id, run


RUN_ID, run = select_run_id(EXPERIMENT_NAME, RUN_ID)
params = run.data.params
metrics = run.data.metrics
best_mse = metrics.get("best_mse")

print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Run ID: {RUN_ID}")
print(f"Run name: {run.info.run_name}")
print(f"best_mse: {best_mse}")

In [ ]:
model_uri = f"runs:/{RUN_ID}/{MODEL_ARTIFACT}"
model = mlflow.pytorch.load_model(model_uri, map_location=device).to(device).eval()

n_classes = int(params.get("n_classes", N_CLASSES))
null_id = int(params.get("null_id", n_classes))
data_transform = params.get("data_transform", "default")

print(f"Loaded: {model_uri}")
print(f"device={device}, n_classes={n_classes}, null_id={null_id}, data_transform={data_transform}")

In [ ]:
def samples_to_display(samples: torch.Tensor, data_transform: str) -> torch.Tensor:
    images = samples.detach().cpu()
    if data_transform == "default":
        mean = torch.tensor(CIFAR10_MEAN, dtype=images.dtype).view(1, -1, 1, 1)
        std = torch.tensor(CIFAR10_STD, dtype=images.dtype).view(1, -1, 1, 1)
        images = images * std + mean
    elif images.min() < 0:
        images = (images + 1) / 2
    return images.clamp(0, 1)


def plot_cfg_sweep(rows: list[torch.Tensor], cfg_scales: list[float], data_transform: str):
    n_rows = len(rows)
    n_cols = rows[0].shape[0]
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(1.45 * n_cols, 1.55 * n_rows),
        squeeze=False,
        constrained_layout=True,
    )

    for row_idx, (scale, samples) in enumerate(zip(cfg_scales, rows)):
        images = samples_to_display(samples, data_transform)
        for col_idx in range(n_cols):
            ax = axes[row_idx][col_idx]
            ax.imshow(images[col_idx].permute(1, 2, 0).numpy())
            ax.set_xticks([])
            ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(CIFAR10_LABELS[col_idx], fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(f"CFG {scale:g}", fontsize=10)

    fig.suptitle(f"CFG sweep for run {RUN_ID[:8]} ({ODE_SOLVER_NAME}, {ODE_STEPS} steps, seed {SEED})")
    return fig

In [ ]:
ode_solver = get_ode_solver_from_name(ODE_SOLVER_NAME)
labels = torch.arange(n_classes, dtype=torch.long, device=device)

rows = []
for scale in CFG_SCALES:
    print(f"Sampling CFG scale {scale:g}...")
    samples = sample_conditional(
        model=model,
        y=labels,
        image_shape=IMAGE_SHAPE,
        ode_solver=ode_solver,
        n_steps=ODE_STEPS,
        guidance_scale=float(scale),
        null_id=null_id,
        device=device,
        seed=SEED,
        clamp_mode=CLAMP_MODE,
    )
    rows.append(samples.cpu())

fig = plot_cfg_sweep(rows, CFG_SCALES, data_transform)
display(fig)

In [ ]:
SAVE_GRID = True
LOG_GRID_TO_MLFLOW = False
EVAL_EXPERIMENT_NAME = "CFG strength sweeps"

out_dir = PROJECT_ROOT / "notebooks" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
grid_path = out_dir / f"cfg_strength_sweep_{RUN_ID[:8]}.png"

if SAVE_GRID:
    fig.savefig(grid_path, dpi=180)
    print(f"Saved {grid_path}")

if LOG_GRID_TO_MLFLOW:
    mlflow.set_experiment(EVAL_EXPERIMENT_NAME)
    with mlflow.start_run(run_name=f"cfg_sweep::{RUN_ID[:8]}"):
        mlflow.log_param("source_run_id", RUN_ID)
        mlflow.log_param("model_uri", model_uri)
        mlflow.log_param("cfg_scales", ",".join(map(str, CFG_SCALES)))
        mlflow.log_param("ode_solver", ODE_SOLVER_NAME)
        mlflow.log_param("ode_steps", ODE_STEPS)
        mlflow.log_param("seed", SEED)
        mlflow.log_artifact(str(grid_path), artifact_path="cfg_sweeps")
        print("Logged CFG sweep image to MLflow.")